# Effects of Quantization on Linear Probe Layer Selection

This notebook fits identical linear probes to quantized and full-bit GPT-2-small models and checks on what layer the linear probles become effective.

**Setup.** Two variants of GPT-2-small are loaded: a full-bit (float32) baseline and an
8-bit (bitsandbytes `Linear8bitLt`) quantized version. For each of 3 classification
challenges (`cities`, `sp_en_trans`, `larger_than`, from the geometry-of-truth dataset
also used in `part31_linear_probes`), a probe is trained at every transformer block's
output (last-token / period-token activations) and evaluated on a held-out split. Two
probe types are used: a difference-of-means probe and an L2-regularized logistic
regression probe, both read off the same activations.

All dataset construction, activation extraction, probe fitting, and sweep orchestration
lives in `probe_datasets.py`, `model_loading.py`, `activations.py`, `probes.py`, and
`sweep.py` in this folder. This notebook only calls into those scripts and visualizes
the resulting per-layer accuracies.

In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib widget

In [8]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd() 
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from IPython.display import display
from plotly.subplots import make_subplots

from model_loading import MODEL_CONFIGS, load_full_bit_model, load_quantized_model, load_tokenizer
from probe_datasets import DATASET_NAMES, load_challenge_datasets
from sweep import run_layer_sweep

device = "cuda" if t.cuda.is_available() else "cpu"

# ---------------------------------------------------------------------------
# Sweep scale for the whole notebook. "quick" subsamples each dataset for a fast
# sanity pass; "full" uses every statement. Every experiment below reads this.
SIZE = "full"  # "quick" or "full"
DATASET_SIZE = {"quick": 200, "full": None}[SIZE]

# Which base model to probe. "llama2-13b" is gated on HuggingFace: HF_TOKEN in .env
# must belong to an account that has been granted access to meta-llama/Llama-2-13b-hf.
MODEL = "llama2-13b"  # "gpt2" or "llama2-13b"
NUM_LAYERS = MODEL_CONFIGS[MODEL]["num_layers"]
D_MODEL = MODEL_CONFIGS[MODEL]["d_model"]  

BATCH_SIZE = 32
TRAIN_FRAC = 0.8
LAYERS = list(range(NUM_LAYERS))

print(f"SIZE={SIZE!r}, MODEL={MODEL!r}, device={device!r}, NUM_LAYERS={NUM_LAYERS}, D_MODEL={D_MODEL}")

SIZE='full', MODEL='llama2-13b', device='cuda', NUM_LAYERS=40, D_MODEL=5120


In [3]:
datasets = load_challenge_datasets(size=DATASET_SIZE)

summary = pd.DataFrame(
    {
        "Dataset": DATASET_NAMES,
        "N statements": [len(datasets[n]) for n in DATASET_NAMES],
        "N train": [int(len(datasets[n]) * TRAIN_FRAC) for n in DATASET_NAMES],
        "N test": [len(datasets[n]) - int(len(datasets[n]) * TRAIN_FRAC) for n in DATASET_NAMES],
        "N positive": [int(datasets[n]["label"].sum()) for n in DATASET_NAMES],
        "N negative": [int((1 - datasets[n]["label"]).sum()) for n in DATASET_NAMES],
    }
)
display(summary)

,Dataset,N statements,N train,N test,N positive,N negative
0,cities,1496,1196,300,748,748
1,sp_en_trans,354,283,71,177,177
2,larger_than,1980,1584,396,990,990


In [4]:
tokenizer = load_tokenizer(MODEL)

if "full_model" not in dir():
    full_model = load_full_bit_model(MODEL, device=device)
if "quant_model" not in dir():
    quant_model = load_quantized_model(MODEL) 

model_variants = {
    "full_bit": (full_model, tokenizer),
    "quantized": (quant_model, tokenizer),
}

for name, (model, _) in model_variants.items():
    n_params = sum(p.numel() for p in model.parameters())
    dtypes = {str(p.dtype) for p in model.parameters()}
    print(f"{name}: {n_params:,} params, dtypes={dtypes}")

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

full_bit: 13,015,864,320 params, dtypes={'torch.float32'}
quantized: 13,015,864,320 params, dtypes={'torch.float16', 'torch.int8'}


In [5]:
t.manual_seed(42)

results = run_layer_sweep(
    datasets=datasets,
    model_variants=model_variants,
    layers=LAYERS,
    train_frac=TRAIN_FRAC,
    seed=42,
    batch_size=BATCH_SIZE,
)


Training probes:   0%|          | 0/480 [00:00<?, ?it/s]

In [9]:
# Saving to disk
results_path = Path(f"results/{MODEL}/{SIZE}-{DATASET_SIZE}/results.pkl")
results_path.parent.mkdir(parents=True)
results.to_pickle(results_path)


## Reading from disk
# results = pd.read_pickle(results_path)

In [11]:
probe_types = sorted(results["probe_type"].unique())
variant_names = list(model_variants.keys())

print(f"{len(results)} rows: {len(DATASET_NAMES)} datasets x {len(variant_names)} variants x "
      f"{len(probe_types)} probe types x {len(LAYERS)} layers")
print(f"probe_types={probe_types}, variant_names={variant_names}")

480 rows: 3 datasets x 2 variants x 2 probe types x 40 layers
probe_types=['diff_of_means', 'logistic'], variant_names=['full_bit', 'quantized']


## Test accuracy by layer, one table per dataset

For each dataset, rows are layers and columns are `{probe_type}: {full_bit, quantized, Δ}`,
where `Δ = quantized − full_bit`. `n_train`/`n_test` are fixed per dataset (see the dataset
table above) and are not repeated per row here.

In [12]:
def build_layer_table(results: pd.DataFrame, dataset_name: str, probe_types: list[str], layers: list[int]) -> pd.DataFrame:
    """Wide table: index=layer, columns={probe_type}: {full_bit, quantized, Δ} of test_acc."""
    sub = results[results["dataset"] == dataset_name]
    table = pd.DataFrame(index=pd.Index(layers, name="layer"))
    for probe_type in probe_types:
        probe_sub = sub[sub["probe_type"] == probe_type]
        fb = probe_sub[probe_sub["model_variant"] == "full_bit"].set_index("layer")["test_acc"]
        qz = probe_sub[probe_sub["model_variant"] == "quantized"].set_index("layer")["test_acc"]
        table[f"{probe_type}: full_bit"] = fb
        table[f"{probe_type}: quantized"] = qz
        table[f"{probe_type}: Δ"] = qz - fb
    return table.round(3)


for dataset_name in DATASET_NAMES:
    print(f"\n{dataset_name}")
    display(build_layer_table(results, dataset_name, probe_types, LAYERS))


cities


,diff_of_means: full_bit,diff_of_means: quantized,diff_of_means: Δ,logistic: full_bit,logistic: quantized,logistic: Δ
layer,,,,,,
0,0.510,0.510,0.000,0.337,0.477,0.140
1,0.510,0.510,0.000,0.213,0.443,0.230
2,0.510,0.510,0.000,0.190,0.337,0.147
3,0.493,0.487,-0.007,0.470,0.453,-0.017
4,0.520,0.497,-0.023,0.823,0.823,0.000
5,0.640,0.640,0.000,0.867,0.880,0.013
6,0.593,0.610,0.017,0.873,0.877,0.003
7,0.663,0.677,0.013,0.940,0.937,-0.003
8,0.603,0.617,0.013,0.990,0.987,-0.003



sp_en_trans


,diff_of_means: full_bit,diff_of_means: quantized,diff_of_means: Δ,logistic: full_bit,logistic: quantized,logistic: Δ
layer,,,,,,
0,0.535,0.535,0.000,0.549,0.479,-0.070
1,0.535,0.535,0.000,0.465,0.408,-0.056
2,0.535,0.549,0.014,0.423,0.423,0.000
3,0.549,0.549,0.000,0.507,0.437,-0.070
4,0.563,0.577,0.014,0.690,0.620,-0.070
5,0.634,0.676,0.042,0.789,0.803,0.014
6,0.831,0.845,0.014,0.887,0.887,0.000
7,0.817,0.817,0.000,0.972,0.972,0.000
8,0.845,0.831,-0.014,0.986,0.986,0.000



larger_than


,diff_of_means: full_bit,diff_of_means: quantized,diff_of_means: Δ,logistic: full_bit,logistic: quantized,logistic: Δ
layer,,,,,,
0,0.462,0.462,0.000,0.992,0.967,-0.025
1,0.462,0.462,0.000,0.992,0.982,-0.010
2,0.462,0.462,0.000,0.992,0.982,-0.010
3,0.816,0.593,-0.222,0.990,0.985,-0.005
4,0.621,0.856,0.235,0.990,0.990,0.000
5,0.874,0.871,-0.003,1.000,1.000,0.000
6,0.899,0.881,-0.018,1.000,1.000,0.000
7,0.545,0.720,0.174,1.000,1.000,0.000
8,0.581,0.462,-0.119,1.000,1.000,0.000


## Layer sweep: test accuracy by layer, model variant, probe type

Grid of `probe_type` (rows) x `dataset` (columns); each panel plots test accuracy against
layer for both model variants.

In [17]:
variant_colors = {"full_bit": "#4c72b0", "quantized": "#dd8452"}

fig = make_subplots(
    rows=len(probe_types),
    cols=len(DATASET_NAMES),
    subplot_titles=[f"{d}" for d in DATASET_NAMES] * 1,
    row_titles=probe_types,
    shared_yaxes=True,
)

for row, probe_type in enumerate(probe_types, start=1):
    for col, dataset_name in enumerate(DATASET_NAMES, start=1):
        for variant_name in variant_names:
            sub = results[
                (results["probe_type"] == probe_type)
                & (results["dataset"] == dataset_name)
                & (results["model_variant"] == variant_name)
            ].sort_values("layer")
            fig.add_trace(
                go.Scatter(
                    x=sub["layer"],
                    y=sub["test_acc"],
                    mode="lines+markers",
                    name=variant_name,
                    legendgroup=variant_name,
                    showlegend=(row == 1 and col == 1),
                    line=dict(color=variant_colors[variant_name]),
                ),
                row=row,
                col=col,
            )
        fig.update_xaxes(title_text="Layer" if row == len(probe_types) else "", row=row, col=col)
        fig.update_yaxes(title_text="Test acc" if col == 1 else "", range=[-0.05, 1.05], row=row, col=col)

fig.update_layout(
    title=f"Probe test accuracy by layer (MODEL={MODEL!r}, SIZE={SIZE!r})",
    height=350 * len(probe_types),
    width=350 * len(DATASET_NAMES) + 100,
)
fig.show()

## Quantized minus full-bit test accuracy, by layer

For each `(dataset, probe_type)` pair, a heatmap over layers of
`test_acc[quantized] - test_acc[full_bit]`.

In [14]:
pivot = results.pivot_table(index=["dataset", "probe_type", "layer"], columns="model_variant", values="test_acc").reset_index()
pivot["delta_quantized_minus_full"] = pivot["quantized"] - pivot["full_bit"]
pivot["row_label"] = pivot["dataset"] + " / " + pivot["probe_type"]

heat = pivot.pivot(index="row_label", columns="layer", values="delta_quantized_minus_full")
heat = heat.reindex(sorted(heat.index))

max_abs = float(heat.abs().to_numpy().max())
fig = go.Figure(
    go.Heatmap(
        z=heat.values,
        x=[str(c) for c in heat.columns],
        y=heat.index,
        colorscale="RdBu",
        zmid=0,
        zmin=-max_abs,
        zmax=max_abs,
        colorbar=dict(title="Δ test acc"),
        text=[[f"{v:.3f}" for v in row] for row in heat.values],
        texttemplate="%{text}",
    )
)
fig.update_layout(
    title="Quantized − full-bit test accuracy, per layer",
    xaxis_title="Layer",
    height=120 + 40 * len(heat.index),
    width=120 + 60 * len(heat.columns),
)
fig.show()

## Best layer summary

One row per (dataset, probe_type); full_bit and quantized best layers/accuracies sit side by side.

In [15]:
rows = []
for dataset_name in DATASET_NAMES:
    for probe_type in probe_types:
        sub = results[(results["dataset"] == dataset_name) & (results["probe_type"] == probe_type)]
        fb = sub[sub["model_variant"] == "full_bit"].pipe(lambda d: d.loc[d["test_acc"].idxmax()])
        qz = sub[sub["model_variant"] == "quantized"].pipe(lambda d: d.loc[d["test_acc"].idxmax()])
        rows.append(
            {
                "dataset": dataset_name,
                "probe_type": probe_type,
                "full_bit: best layer": int(fb["layer"]),
                "full_bit: test_acc": round(fb["test_acc"], 3),
                "quantized: best layer": int(qz["layer"]),
                "quantized: test_acc": round(qz["test_acc"], 3),
                "Δ test_acc": round(qz["test_acc"] - fb["test_acc"], 3),
            }
        )

best_layer_df = pd.DataFrame(rows)
display(best_layer_df)

,dataset,probe_type,full_bit: best layer,full_bit: test_acc,quantized: best layer,quantized: test_acc,Δ test_acc
0,cities,diff_of_means,13,0.897,12,0.890,-0.007
1,cities,logistic,11,0.997,11,0.990,-0.007
2,sp_en_trans,diff_of_means,11,0.972,12,0.986,0.014
3,sp_en_trans,logistic,15,1.000,14,1.000,0.000
4,larger_than,diff_of_means,12,0.997,12,0.985,-0.013
5,larger_than,logistic,5,1.000,5,1.000,0.000
